# 13 · Evaluate generated samples (CPU runtime, no GPU quota)

**Runtime → Change runtime type → CPU.** Computes the metric tables (`scripts/evaluate.py`) for the experiments listed in
`EVAL` and writes `per_sample.csv` / `per_image.csv` to `MyDrive/CS787/eval/<experiment>/<job>/`. Already-evaluated
jobs are skipped, so rerun after each generation session. About 5–10 s per sample for the full metric set on Colab CPU;
use `GROUPS = "patch,copy,structure"` for a fast pass (no pretrained networks).

In [ ]:
# ==== SETTINGS (edit once) ====================================================================
CODE_SOURCE   = "github"                                      # "github" or "drive"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
REPO_SUBDIR   = ""                                            # project files are at the repo root
DRIVE_CODE    = "/content/drive/MyDrive/CS787/code/Project"   # used when CODE_SOURCE == "drive"
DRIVE_ROOT    = "/content/drive/MyDrive/CS787"                # outputs, checkpoints, tables, env logs
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
# If the GitHub repo is private: add a Colab secret named GH_TOKEN (key icon in the left bar)
# holding a fine-grained, read-only token for that repo.

NOTEBOOK    = "13_evaluate"
GROUPS      = "paper,patch,copy,structure"
MAX_SAMPLES = 50
# experiment folder -> manifests that contain its input images (ids must not collide, so list only the relevant ones)
EVAL = {
    "phase4_reproduction": ["uncond_sinddm15"],
    "phase7_ood":          ["ood_setA", "synthetic_setB"],
    "phase8_ours":         ["sinddm_ood3", "lora8"],
    "phase8_sinddm":       ["uncond_sinddm15", "sinddm_ood3"],
    "phase8_lora":         ["lora8"],
    "phase9_ablations":    ["ablation_subset"],
    "phase9_ab6_ann":      ["ablation_subset"],
}

In [ ]:
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
for sub in ("outputs", "checkpoints", "tables", "env_logs", "data"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

PROJECT = "/content/project"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
if CODE_SOURCE == "github":
    url = GITHUB_REPO
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # public repo: no token needed
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, url, "/content/repo"], check=True)
    shutil.copytree(os.path.join("/content/repo", REPO_SUBDIR), PROJECT,
                    ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
else:
    shutil.copytree(DRIVE_CODE, PROJECT, ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
print("project code at", PROJECT, "->", sorted(os.listdir(PROJECT)))

In [ ]:
!pip install -q lpips pytorch-fid pyiqa opencv-python-headless scikit-image scipy pandas
!pip install -q -e {PROJECT}
sys.path.insert(0, f"{PROJECT}/src")
# the natural-image sets are third-party and not in the repo: fetch them and check against the manifests
SID_ROOT = "/content/efficient-SID-code"
if not os.path.exists(SID_ROOT):
    subprocess.run(["git", "clone", "-q", "https://github.com/haojunqiu/efficient-SID-code", SID_ROOT], check=True)
r = subprocess.run([sys.executable, "scripts/prepare_natural.py", "--verify", "--sets", "uncond/sinddm15", "--sid-root", SID_ROOT],
                   cwd=PROJECT, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
assert r.returncode == 0, "natural images do not match the manifest"

In [ ]:
OUT = f"{DRIVE_ROOT}/outputs"
for exp, mans in EVAL.items():
    if not os.path.isdir(f"{OUT}/{exp}"):
        print(exp, "-> no outputs yet"); continue
    cmd = [sys.executable, "scripts/evaluate.py", f"{OUT}/{exp}", "--groups", GROUPS, "--max-samples", str(MAX_SAMPLES),
           "--eval-root", f"{DRIVE_ROOT}/eval"]
    for m in mans:
        cmd += ["--manifest", f"{PROJECT}/data/manifests/{m}.csv"]
    print("=====", exp)
    subprocess.run(cmd, cwd=PROJECT)

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
!pip freeze > {DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt
print("saved", f"{DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt")